# M1C2 - Topic 7: Data Science with Pandas and Kaggle

In this notebook we revisit every concept from the Practical Lab 1 1 — but this time using
**industry-standard tools**: the **Kaggle** platform for real-world data and the **pandas**
library for data manipulation.

We use the **same FRIENDS TV series** theme, but instead of hand-typing 25 episodes we
download the **complete official dataset** (all 10 seasons, 227 episodes) directly from
Kaggle.

## Notebook Content

1. **Loading Data from Kaggle** — Download the FRIENDS dataset with `kagglehub` and load it into a pandas `DataFrame`
2. **Exploring Data with Pandas** — Replace lists, dicts, and sets with `DataFrame` / `Series` operations
3. **Filtering & Transformation** — Replace `if/elif/else` and `for` loops with boolean indexing and `groupby`
4. **Text Preprocessing** — Replace manual character-by-character functions with `.str` methods and `apply()`
5. **Bag of Words** — Compute word frequencies using pandas instead of manual dictionaries
6. **Text Classification** — Same keyword-based classifier, powered by pandas
7. **Evaluation** — Accuracy, confusion matrix, precision & recall with `pd.crosstab`
8. **Classifier Variants** — Term-frequency and weighted scoring, compared side by side

## Rationale

In the Practical Lab 1 you built a complete text-classification pipeline **from scratch**
using only Python built-ins — lists, dicts, sets, loops, and functions. That exercise
showed you *what* libraries like pandas do under the hood.

Now it is time to see how **real data scientists** work:

| Practical Lab 1 (manual) | Topic 7 (official) |
|---|---|
| Hand-typed list of 25 dicts | Real Kaggle dataset with 227 episodes |
| `for` loops to filter / count | Boolean indexing, `.groupby()`, `.value_counts()` |
| Character-by-character cleaning | `.str` accessor and `re` module |
| Manual frequency dicts | `pd.Series.value_counts()` |
| Nested loops for confusion matrix | `pd.crosstab()` |

By the end of this notebook you will be comfortable loading, cleaning, exploring,
and classifying real data the way it is done in industry.

## Dataset

We use the **Friends TV Show — All Seasons and Episodes Data** from Kaggle
([kaggle.com/datasets/ruchi798/friends-tv-show-all-seasons-and-episodes-data](https://www.kaggle.com/datasets/ruchi798/friends-tv-show-all-seasons-and-episodes-data)).
It contains all 227 regular episodes across 10 seasons with columns such as
*Title*, *Summary*, *Episode*, *Date*, *U.S. viewers*, *Directed by*, and more.

We will classify each episode's **Summary** into the same four theme categories
used in the Practical Lab 1: `Romance`, `Career`, `Family`, `Friendship`.

---

## 1. Setting Up the Environment

In [ ]:
!pip install --quiet kagglehub pandas

In [ ]:
import kagglehub
import pandas as pd
import os
import re
from collections import Counter

---

## 2. Loading Data from Kaggle

`kagglehub` is the **official Kaggle Python package** for downloading datasets
programmatically. It caches datasets locally so you only download once.

In [ ]:
path = kagglehub.dataset_download("ruchi798/friends-tv-show-all-seasons-and-episodes-data")

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(f"Dataset downloaded to: {path}")
print(f"CSV files found: {csv_files}")

In [ ]:
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

---

## 3. Exploring the Dataset with Pandas

In the Practical Lab 1 you stored episodes as a **list of dictionaries** and
used **for loops** to inspect them. Pandas replaces all of that with a single
object — the **DataFrame**.

### 3.1 DataFrame Basics

A `DataFrame` is a two-dimensional table — like a spreadsheet or a SQL table.
Each column is a `Series` (similar to a Python list), and each row is a record
(similar to a dictionary).

In [ ]:
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
print("Column data types:")
print(df.dtypes)

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

### 3.2 Data Cleaning

Real-world data is rarely clean. This dataset has:
- **2 special episodes** that are clip shows, not regular episodes
- **Multi-part episodes** encoded with newlines in the `Episode` column (e.g. `"02-12\n02-13"`)
- The `U.S. viewers` column stored as text like `"21.5 million"`

With pandas, cleaning all of this takes just a few lines.
- Episode number changed to two columns: season and episode
- Drop rows with empty columns
- Remove 'million' from viewers column

In [ ]:
df = df[~df["Episode"].str.contains("Special", na=False)].copy()

df["Episode"] = df["Episode"].str.split("\n").str[0]

df["season"] = df["Episode"].str.split("-").str[0].astype(int)
df["episode_num"] = df["Episode"].str.split("-").str[1].astype(int)

df["viewers_millions"] = (
    df["U.S. viewers"]
    .str.replace(" million", "", regex=False)
    .apply(pd.to_numeric, errors="coerce")
)

df = df.dropna(subset=["Summary"])

print(f"Cleaned dataset: {len(df)} episodes across seasons {df['season'].min()}–{df['season'].max()}")
print(f"\nColumns after cleaning: {list(df.columns)}")

In [ ]:
df[["season", "episode_num", "Title", "Summary"]].head(10)

### 3.3 Series — the Pandas Equivalent of a List

In the Practical Lab 1 we used a plain Python **list** to store titles:
```python
titles = ["The Pilot", "The One with the Sonogram at the End", ...]
```

With pandas we simply access a column — the result is a `Series`, which
behaves like a list but with many extra capabilities.

In [ ]:
titles = df["Title"]

print("Type:", type(titles))
print("Length:", len(titles))
print("First title:", titles.iloc[0])
print("Last title:", titles.iloc[-1])

In [ ]:
print("First 5 titles:")
print(titles.head().to_string())

print("\nLast 5 titles:")
print(titles.tail().to_string())

print("\nSlicing [10:15]:")
print(titles.iloc[10:15].to_string())

### 3.4 Unique Values — the Pandas Equivalent of a Set

In the Practical Lab 1 we used Python **sets** to find distinct categories:
```python
categories = set(example_list)   # {"Romance", "Career", ...}
seasons = {1, 2, 3, ...}
```

Pandas offers `.unique()`, `.nunique()`, and `.value_counts()` — all in one line.

In [ ]:
print("Unique seasons:", sorted(df["season"].unique()))
print("Number of seasons:", df["season"].nunique())

print("\nEpisodes per season:")
print(df["season"].value_counts().sort_index())

In [ ]:
print("Unique directors:", df["Directed by"].nunique())
print("\nTop 5 directors by episode count:")
print(df["Directed by"].value_counts().head())

---

## 4. Filtering and Transformation

In the Practical Lab 1 we used **`if/elif/else`** inside **`for` loops** to filter
and count data. Pandas replaces both with **boolean indexing** and **`.groupby()`**.

### 4.1 Boolean Indexing — Replaces `if/elif/else` in Loops

Instead of looping through every row and checking a condition, we write the
condition once and pandas applies it to the entire column at once.

Filter for example **Season 1** episodes

In [ ]:
season_1 = df[df["season"] == 1]
print(f"Season 1 episodes: {len(season_1)}")
season_1[["episode_num", "Title", "viewers_millions"]].head()

Filter the most viewed episodes starting from season 5 onwards:

In [ ]:
popular = df[
    (df["viewers_millions"] > 30) &
    (df["season"] >= 5)
]
print(f"Episodes with 30M+ viewers from season 5 onward: {len(popular)}")
popular[["season", "episode_num", "Title", "viewers_millions"]]

Order the most viewed episodes starting from season 5

In [ ]:
popular_sorted = popular.sort_values("viewers_millions", ascending=False)
popular_sorted[["season", "episode_num", "Title", "viewers_millions"]]

### 4.2 GroupBy — Replaces Nested `for` Loops

In the Practical Lab 1 we built a category-by-season frequency table using
**three nested loops**. Pandas does this in **one line** with `.groupby()`.

Show Mean, Min, Max, Count information from each season **views** statistic.
Which season was most viewed?

In [ ]:
viewer_stats = df.groupby("season")["viewers_millions"].agg(["mean", "min", "max", "count"])
print("Views statistics by season:")
print(viewer_stats.round(1))

Order the most viewed seasons

In [ ]:
season_avg = df.groupby("season")["viewers_millions"].mean().sort_values(ascending=False)
print("Seasons ranked by average viewership (highest to lowest):")
print(season_avg.round(2))

In [ ]:
# TODO: Repeat the same approach but now with the most rated episodes
df["rating"] = df["Rating/Share"].str.split("/").str[0].apply(pd.to_numeric, errors="coerce")

season_avg_rating = df.groupby("season")["rating"].mean().sort_values(ascending=False)
print("Seasons ranked by average rating (highest to lowest):")
print(season_avg_rating.round(2))

In [ ]:
rating_stats = df.groupby("season")["rating"].agg(["mean", "min", "max", "count"]).sort_values("mean", ascending=False)
print("Rating statistics by season (sorted by mean rating):")
print(rating_stats.round(1))

#### Selecting Rows by Label with `.loc[]`

Pandas provides **`.loc[]`** to select rows and columns **by label** rather than
by integer position. While `.iloc[]` uses numeric indices (0, 1, 2, …),
`.loc[]` uses the actual index labels — which can be strings, dates, or any
other hashable type.

The following code builds a director × season table and then uses `.loc[]`
to extract only the rows corresponding to the **top 5 directors**:

1. **`df.groupby(["Directed by", "season"]).size().unstack(fill_value=0)`** — groups every episode by director and season, counts the number of episodes in each combination, then *unstacks* the season level into columns. The result is a table where each row is a director and each column is a season.
2. **`df["Directed by"].value_counts().head(5).index`** — finds the 5 directors who directed the most episodes overall, returning just their names as an `Index`.
3. **`director_seasons.loc[top_directors]`** — uses `.loc[]` to select only the rows whose index label matches one of those 5 director names, filtering the large table down to just the rows we care about.

In [ ]:
director_seasons = df.groupby(["Directed by", "season"]).size().unstack(fill_value=0)
print("Top 5 directors — episodes per season:")
top_directors = df["Directed by"].value_counts().head(5).index
print(director_seasons.loc[top_directors])

### 4.3 Searching with `.str.contains()` — Replaces `while` Loops

In the Practical Lab 1 we used a `while` loop to iteratively broaden a keyword
search until we found a match. With pandas, a single `.str.contains()` call
scans every summary at once.

In [ ]:
search_term = "wedding"
matches = df[
    df["Summary"].str.lower().str.contains(search_term, na=False)
]
print(f"Episodes whose summary contains '{search_term}': {len(matches)}")
matches[["season", "episode_num", "Title"]].head(10)

### 4.4 Frequency Table with `pd.crosstab()` — Replaces Nested Counting

In the Practical Lab 1 we built a category-by-season frequency table using
nested loops and a dictionary of dictionaries. `pd.crosstab` does this in
one line.

Here we build a summary-length × season table to see how episode descriptions
vary across seasons.

In [ ]:
df["summary_length"] = df["Summary"].str.len()
df["length_bin"] = pd.cut(
    df["summary_length"],
    bins=[0, 100, 200, 300, 1200],
    labels=["short (<100)", "medium (100-200)", "long (200-300)", "very long (300+)"],
)

freq = pd.crosstab(df["length_bin"], df["season"])
print("Summary length × Season frequency table:")
print(freq)

---

## 5. Text Preprocessing with Pandas

In the Practical Lab 1 we wrote **four separate functions** (`clean_text`,
`tokenize`, `remove_stop_words`, `preprocess`) and looped over every
episode to apply them.

With pandas we use the **`.str` accessor** for vectorized string operations
and **`.apply()`** to run custom functions across the entire column at once.

Recall the four preprocessing steps:
1. **Lowercase** — so *"Love"* and *"love"* are treated identically
2. **Remove punctuation** — so *"romance."* matches *"romance"*
3. **Tokenize** — split into individual words
4. **Remove stop words** — discard common words like *"the"*, *"is"*, *"and"*

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "not", "no", "nor",
    "it", "its", "he", "she", "they", "them", "his", "her", "their",
    "this", "that", "these", "those", "who", "whom", "which", "what",
    "when", "where", "how", "all", "each", "every", "both", "few",
    "more", "most", "other", "some", "such", "only", "own", "same",
    "than", "too", "very", "just", "about", "up", "out", "so", "if",
    "from", "into", "through", "during", "before", "after", "above",
    "as", "then", "once", "here", "there", "also", "over",
}


def preprocess(text):
    """Lowercase, remove punctuation, tokenize, and drop stop words."""
    text = re.sub(r"[^\w\s]", "", text.lower())
    return [w for w in text.split() if w not in STOP_WORDS]

In [ ]:
sample_text = df["Summary"].iloc[0]
print("Original :", sample_text)
print("Lowercase:", sample_text.lower())
print("Cleaned  :", re.sub(r"[^\w\s]", "", sample_text.lower()))
print("Tokens   :", re.sub(r"[^\w\s]", "", sample_text.lower()).split())
print("Filtered :", preprocess(sample_text))

In [ ]:
df["tokens"] = df["Summary"].apply(preprocess)

print(f"Tokenized {len(df)} episode summaries.")
df[["Title", "tokens"]].head()

Compare the difference: in the Practical Lab 1, preprocessing the entire
dataset required a **`for` loop** that called `preprocess()` on each episode
one by one. With pandas, a single **`.apply(preprocess)`** call does the
same work across all 227 episodes in one expression.

---

## 6. Bag of Words — Word Frequency Analysis

A *Bag of Words* converts text into a numerical representation by counting
how often each word appears, ignoring grammar and word order.

In the Practical Lab 1 we built `word_frequencies()` with a manual dictionary
and sorted with a bubble sort. With pandas, `.explode()` + `.value_counts()`
achieves the same result in two lines.

In [ ]:
all_tokens = df["tokens"].explode()
word_freq = all_tokens.value_counts()

print(f"Vocabulary size: {len(word_freq)} unique words\n")
print("Top 20 most frequent words across all episodes:")
for word, count in word_freq.head(20).items():
    bar = "#" * min(count, 60)
    print(f"  {word:<16} {count:>4}  {bar}")

In [ ]:
print("Top 10 words per season (seasons 1, 5, 10):\n")
for s in [1, 5, 10]:
    season_tokens = df[df["season"] == s]["tokens"].explode()
    top = season_tokens.value_counts().head(10)
    print(f"Season {s}:")
    for word, count in top.items():
        print(f"    {word:<14} {count}")
    print()

---

## 7. Text Classification

We use the **same keyword-based approach** from the Practical Lab 1:
1. Define a **keyword dictionary** with representative words for each theme
2. **Score** each episode summary by counting keyword hits
3. **Predict** the theme with the highest score

The four theme categories remain the same: **Romance**, **Career**, **Family**, **Friendship**.

The classification logic is identical — what changes is how we apply it to
the dataset (pandas `.apply()` instead of a manual `for` loop).

### 7.1 Keyword Dictionary

This is the **same keyword dictionary** from the Practical Lab 1.
Each category is associated with a set of representative words.
In a real ML pipeline these would be **learned automatically** from the data;
here we select them manually to illustrate the concept.

In [ ]:
KEYWORDS = {
    "Romance": {
        "love", "kiss", "romance", "romantic", "couple", "feelings",
        "relationship", "dating", "girlfriend", "boyfriend", "engaged",
        "propose", "proposal", "devotion", "passionate", "confess",
        "reunite", "break", "heartbreak", "flirting",
    },
    "Career": {
        "job", "career", "work", "boss", "promotion", "office",
        "interview", "audition", "chef", "hire", "resume",
        "quit", "fired", "position", "business", "advertising",
        "stockbroker", "writer", "actor", "rejection",
    },
    "Family": {
        "baby", "pregnant", "family", "father", "mother", "dad",
        "mom", "parents", "birth", "child", "son", "daughter",
        "childhood", "wedding", "married", "ceremony", "traditions",
        "flashbacks", "house", "labor",
    },
    "Friendship": {
        "friends", "group", "gang", "together", "support", "bond",
        "loyalty", "trivia", "game", "apartment", "roommate",
        "competitive", "argue", "patience", "goodbye", "welcomes",
        "cooperates", "thanksgiving", "holiday", "park",
    },
}

for cat, words in KEYWORDS.items():
    print(f"{cat:>12}: {len(words)} keywords")

### 7.2 Scoring and Classification Functions

These functions are **identical** to the ones from the Practical Lab 1.
The only difference is that we will apply them to the entire DataFrame
at once using `.apply()` instead of a manual `for` loop.

In [ ]:
def score_text(tokens, keyword_dict):
    """Score a token list against every category using set intersection."""
    token_set = set(tokens)
    return {cat: len(token_set & kw) for cat, kw in keyword_dict.items()}


def classify(tokens, keyword_dict):
    """Return (predicted_category, scores) for a token list."""
    scores = score_text(tokens, keyword_dict)
    best = max(scores, key=scores.get)
    return best, scores

In [ ]:
sample_idx = 0
sample_tokens = df["tokens"].iloc[sample_idx]
pred, scores = classify(sample_tokens, KEYWORDS)

print(f"Title    : {df['Title'].iloc[sample_idx]}")
print(f"Summary  : {df['Summary'].iloc[sample_idx]}")
print(f"Tokens   : {sample_tokens}")
print(f"Scores   : {scores}")
print(f"Predicted: {pred}")

### 7.3 Classify the Entire Dataset

In the Practical Lab 1 we wrote:
```python
for ep in episodes:
    pred, _ = classify(ep["description"], KEYWORDS)
    results.append((ep["category"], pred))
```

With pandas, a single `.apply()` call replaces the entire loop — classifying
all 227 episodes at once.

In [ ]:
df["predicted"] = df["tokens"].apply(lambda t: classify(t, KEYWORDS)[0])

print("Classification complete — all 227 episodes classified.")
df[["season", "episode_num", "Title", "predicted"]].head(20)

In [ ]:
print("Predicted theme distribution:")
print(df["predicted"].value_counts())

print("\nThemes per season:")
print(pd.crosstab(df["season"], df["predicted"]))

---

## 8. Evaluate the Classifier

To evaluate the classifier we need **ground-truth labels** — human-assigned
theme categories for each episode.

In the Practical Lab 1, the 25 hand-picked episodes already had labels. Here
we reuse those **same 25 episodes** as our **labeled validation set** and match
them to the Kaggle data by season and episode number.

This is a common real-world pattern: label a small subset, evaluate on it,
then apply the model to the full unlabeled dataset.

In [ ]:
GROUND_TRUTH = {
    (1, 1): "Friendship",
    (1, 2): "Family",
    (1, 7): "Romance",
    (1, 15): "Career",
    (1, 18): "Career",
    (1, 24): "Romance",
    (2, 5): "Friendship",
    (2, 7): "Romance",
    (2, 14): "Romance",
    (3, 2): "Friendship",
    (3, 9): "Friendship",
    (3, 15): "Romance",
    (4, 3): "Career",
    (4, 12): "Friendship",
    (5, 8): "Family",
    (5, 14): "Romance",
    (6, 6): "Friendship",
    (6, 15): "Career",
    (6, 24): "Romance",
    (7, 17): "Career",
    (8, 1): "Family",
    (8, 23): "Family",
    (9, 7): "Family",
    (10, 12): "Family",
    (10, 17): "Romance",
}

df["category"] = df.apply(
    lambda r: GROUND_TRUTH.get((r["season"], r["episode_num"])), axis=1
)

df_labeled = df[df["category"].notna()].copy()
print(f"Labeled episodes matched: {len(df_labeled)} out of {len(GROUND_TRUTH)} expected")
print(f"\nCategory distribution in labeled set:")
print(df_labeled["category"].value_counts())

In [ ]:
print(f"{'Episode':<8} {'Title':<50} {'True':<14} {'Predicted':<14} {'Match'}")
print("-" * 100)
for _, row in df_labeled.iterrows():
    eid = f"S{row['season']:02d}E{row['episode_num']:02d}"
    title = row["Title"][:48]
    true = row["category"]
    pred = row["predicted"]
    mark = "OK" if true == pred else "MISS"
    print(f"{eid:<8} {title:<50} {true:<14} {pred:<14} {mark}")

### Accuracy

In [ ]:
accuracy = (df_labeled["predicted"] == df_labeled["category"]).mean()
print(f"Overall accuracy on labeled subset: {accuracy:.1%}")

Notice that the accuracy on the **real Kaggle summaries** may differ from the
100% we achieved in the Practical Lab 1. This is because the Practical Lab 1
used **hand-written descriptions** carefully crafted to match the keywords,
while the Kaggle summaries are **real-world text** written independently.

This is an important lesson: a model that works perfectly on curated data
may perform differently on real data.

### Confusion Matrix with `pd.crosstab`

In the Practical Lab 1 we built the confusion matrix manually with nested loops
and a dictionary of dictionaries. `pd.crosstab` does the same in one line.

In [ ]:
cm = pd.crosstab(
    df_labeled["category"],
    df_labeled["predicted"],
    rownames=["True"],
    colnames=["Predicted"],
)
print("Confusion Matrix:")
print(cm)

### Precision and Recall

- **Precision**: Of all episodes *predicted* as category X, how many truly are X?
- **Recall**: Of all episodes that truly *are* category X, how many did we predict correctly?

In [ ]:
def precision_recall_from_crosstab(cm):
    """Compute per-category precision and recall from a pandas crosstab."""
    metrics = {}
    for label in cm.index:
        tp = cm.loc[label, label] if label in cm.columns else 0
        predicted_total = cm[label].sum() if label in cm.columns else 0
        actual_total = cm.loc[label].sum()
        prec = tp / predicted_total if predicted_total > 0 else 0.0
        rec = tp / actual_total if actual_total > 0 else 0.0
        metrics[label] = {"precision": prec, "recall": rec}
    return pd.DataFrame(metrics).T


pr = precision_recall_from_crosstab(cm)
print("Per-category Precision and Recall:\n")
print(f"{'Category':>14} {'Precision':>12} {'Recall':>12}")
print("-" * 40)
for label in sorted(set(GROUND_TRUTH.values())):
    if label in pr.index:
        p = pr.loc[label, "precision"]
        r = pr.loc[label, "recall"]
        print(f"{label:>14} {p:>11.1%} {r:>11.1%}")

---

## 9. Exploring a Classifier Variant: Term Frequency (TF)

Our first approach uses **set intersection** — each keyword either matches or
does not. What if we count *every occurrence* of a keyword instead of just
checking its presence? This is a basic **term-frequency** (TF) approach.

Does more complex always mean better? Let's find out.

In [ ]:
def score_text_tf(tokens, keyword_dict):
    """Score using term frequency: count every keyword occurrence, not just unique matches."""
    scores = {}
    for cat, kw in keyword_dict.items():
        scores[cat] = sum(1 for t in tokens if t in kw)
    return scores


def classify_tf(tokens, keyword_dict):
    """Classify using term-frequency scoring."""
    scores = score_text_tf(tokens, keyword_dict)
    return max(scores, key=scores.get), scores


df["predicted_tf"] = df["tokens"].apply(
    lambda t: classify_tf(t, KEYWORDS)[0]
)

accuracy_tf = (df_labeled["predicted"].eq(df_labeled["category"]).mean())
accuracy_tf_val = (
    df.loc[df["category"].notna(), "predicted_tf"]
    .eq(df.loc[df["category"].notna(), "category"])
    .mean()
)
print(f"Set-based accuracy: {accuracy:.1%}")
print(f"TF-based accuracy : {accuracy_tf_val:.1%}")

In [ ]:
cm_tf = pd.crosstab(
    df_labeled["category"],
    df.loc[df["category"].notna(), "predicted_tf"],
    rownames=["True"],
    colnames=["Predicted"],
)
print("TF Confusion Matrix:")
print(cm_tf)

print()
pr_tf = precision_recall_from_crosstab(cm_tf)
print(f"{'Category':>14} {'Precision':>12} {'Recall':>12}")
print("-" * 40)
for label in sorted(set(GROUND_TRUTH.values())):
    if label in pr_tf.index:
        p = pr_tf.loc[label, "precision"]
        r = pr_tf.loc[label, "recall"]
        print(f"{label:>14} {p:>11.1%} {r:>11.1%}")

### Side-by-side Comparison

Comparing the set-based and TF-based approaches shows whether a more
complex scoring method outperforms a simpler one on **real data**. In practice,
this trade-off between model complexity and performance is a central theme
in Data Science.

In [ ]:
comp = df_labeled[["Title", "category", "predicted"]].copy()
comp["predicted_tf"] = df.loc[df["category"].notna(), "predicted_tf"].values
comp["set_result"] = comp.apply(
    lambda r: "OK" if r["predicted"] == r["category"] else r["predicted"], axis=1
)
comp["tf_result"] = comp.apply(
    lambda r: "OK" if r["predicted_tf"] == r["category"] else r["predicted_tf"], axis=1
)

print(f"{'Title':<50} {'True':<12} {'Set':<12} {'TF':<12}")
print("-" * 88)
for _, row in comp.iterrows():
    print(f"{row['Title'][:48]:<50} {row['category']:<12} {row['set_result']:<12} {row['tf_result']:<12}")

print(f"\nSet-based accuracy: {accuracy:.1%}")
print(f"TF-based accuracy : {accuracy_tf_val:.1%}")

---

## 10. Additional Exercises

Try these on your own to reinforce what you have learned:

1. Write a function `most_confused(cm)` that returns the pair of categories most often
   confused with each other, using the pandas crosstab.
2. Implement a **weighted keyword** classifier where each keyword has
   a weight (e.g. `{"love": 3, "kiss": 2, ...}`) instead of all counting equally.

In [ ]:
# --- Exercise 1: most_confused ---

def most_confused(cm):
    """Return the (true, predicted) pair with the most off-diagonal errors."""
    worst_pair = None
    worst_count = 0
    for true_label in cm.index:
        for pred_label in cm.columns:
            if true_label != pred_label:
                count = cm.loc[true_label, pred_label]
                if count > worst_count:
                    worst_count = count
                    worst_pair = (true_label, pred_label)
    return worst_pair, worst_count


print("=== Exercise 1: Most confused pair ===")

print("\nSet-based confusion matrix:")
pair, count = most_confused(cm)
if pair:
    print(f"  Most confused: true='{pair[0]}' predicted as '{pair[1]}' ({count} time(s))")
else:
    print("  No confusion — perfect classification!")

print("\nTF-based confusion matrix:")
pair_tf, count_tf = most_confused(cm_tf)
if pair_tf:
    print(f"  Most confused: true='{pair_tf[0]}' predicted as '{pair_tf[1]}' ({count_tf} time(s))")
else:
    print("  No confusion — perfect classification!")

In [ ]:
# --- Exercise 2: Weighted keyword classifier ---

WEIGHTED_KEYWORDS = {
    "Romance": {
        "love": 3, "kiss": 3, "romance": 3, "romantic": 3, "passionate": 3,
        "couple": 2, "feelings": 2, "relationship": 2, "dating": 2,
        "confess": 2, "reunite": 2, "devotion": 2,
        "girlfriend": 1, "boyfriend": 1, "engaged": 1, "flirting": 1,
        "propose": 1, "proposal": 1, "break": 1, "heartbreak": 1,
    },
    "Career": {
        "job": 3, "career": 3, "work": 2, "boss": 2, "promotion": 2,
        "office": 2, "interview": 2, "audition": 2, "hire": 2,
        "chef": 2, "resume": 2, "quit": 2, "fired": 2,
        "position": 1, "business": 1, "advertising": 1,
        "stockbroker": 1, "writer": 1, "actor": 1, "rejection": 1,
    },
    "Family": {
        "baby": 3, "pregnant": 3, "family": 3, "birth": 3, "labor": 3,
        "father": 2, "mother": 2, "dad": 2, "mom": 2, "parents": 2,
        "child": 2, "son": 2, "daughter": 2, "childhood": 2,
        "wedding": 1, "married": 1, "ceremony": 1, "traditions": 1,
        "flashbacks": 1, "house": 1,
    },
    "Friendship": {
        "friends": 3, "group": 3, "gang": 3, "together": 2, "support": 2,
        "bond": 2, "loyalty": 2, "roommate": 2,
        "trivia": 1, "game": 1, "apartment": 1, "competitive": 1,
        "argue": 1, "patience": 1, "goodbye": 1, "welcomes": 1,
        "cooperates": 1, "thanksgiving": 1, "holiday": 1, "park": 1,
    },
}


def score_text_weighted(tokens, weighted_keyword_dict):
    """Score a token list using weighted keywords. Each keyword hit adds its weight."""
    scores = {}
    for cat, keyword_weights in weighted_keyword_dict.items():
        total = 0
        for token in tokens:
            if token in keyword_weights:
                total += keyword_weights[token]
        scores[cat] = total
    return scores


def classify_weighted(tokens, weighted_keyword_dict):
    """Classify using weighted keyword scoring."""
    scores = score_text_weighted(tokens, weighted_keyword_dict)
    return max(scores, key=scores.get), scores


df["predicted_w"] = df["tokens"].apply(
    lambda t: classify_weighted(t, WEIGHTED_KEYWORDS)[0]
)

acc_w = (
    df.loc[df["category"].notna(), "predicted_w"]
    .eq(df.loc[df["category"].notna(), "category"])
    .mean()
)

print("=== Exercise 2: Weighted keyword classifier ===\n")
print(f"Set-based accuracy : {accuracy:.1%}")
print(f"TF-based accuracy  : {accuracy_tf_val:.1%}")
print(f"Weighted accuracy  : {acc_w:.1%}")

cm_w = pd.crosstab(
    df_labeled["category"],
    df.loc[df["category"].notna(), "predicted_w"],
    rownames=["True"],
    colnames=["Predicted"],
)
print("\nWeighted Confusion Matrix:")
print(cm_w)

---

## Summary

In this notebook we replicated the entire Practical Lab 1 pipeline using
**industry-standard tools** on the **same FRIENDS dataset** — but this time
downloaded from Kaggle with all 227 episodes instead of 25 hand-typed ones.

| Concept | Practical Lab 1 | This notebook |
|---|---|---|
| Data source | Manually typed 25 episodes | Kaggle dataset (227 episodes) |
| Data structure | `list` of `dict` | `pd.DataFrame` |
| Unique values | `set()` | `.unique()`, `.value_counts()` |
| Filtering | `if` inside `for` | Boolean indexing |
| Counting | Nested `for` loops | `.groupby()`, `pd.crosstab()` |
| Text cleaning | Manual `for char in text` | `re.sub()` + `.str` accessor |
| Apply to all rows | `for ep in episodes` | `.apply()` |
| Confusion matrix | Nested dict of dicts | `pd.crosstab()` |

The **classification logic itself** (keyword scoring, accuracy, precision,
recall) stayed the same — what changed is how we **load**, **store**,
**filter**, and **transform** data. This is the power of pandas: it lets
you focus on the *data science* while it handles the *data engineering*.

We also observed that the classifier may perform **differently on real data**
compared to hand-crafted examples — an important lesson about the gap
between toy datasets and production data.